In [18]:
import random
import pickle
import math
import multiprocessing as mp
from typing import Callable
import pandas as pd
import activations
import numpy as np
import itertools
import random
import copy
import time

## Neat Implementation and set up

In [2]:
class Hyperparameters(object):
    """Hyperparameter settings for the FeedForwardNet object."""

    def __init__(self, **kwargs):
        # distance threshold for speciation
        self.delta_threshold = kwargs.get('delta_threshold', 1.5)
        self.distance_weights = kwargs.get('distance_weights', {
            'excess': 1.0,
            'disjoint': 1.0,
            'weight': 0.4
        })
        self.default_activation = kwargs.get('default_activation', activations.sigmoid)
        self.max_fitness = kwargs.get('max_fitness', float('inf'))
        self.max_generations = kwargs.get('max_generations', float('inf'))
        self.max_fitness_history = kwargs.get('max_fitness_history', 30)
        self.breed_probabilities = kwargs.get('breed_probabilities', {
            'asexual': 0.5,
            'sexual': 0.5
        })
        self.mutation_probabilities = kwargs.get('mutation_probabilities', {
            'node': 0.01,
            'edge': 0.09,
            'weight_perturb': 0.4,
            'weight_set': 0.1,
            'bias_perturb': 0.3,
            'bias_set': 0.1
        })
        self.distributions_parameters = kwargs.get('distributions_parameters', {
            'uniform_range': 1,
            'gauss_sigma': 1
        })
        self.interspecies_mating = kwargs.get('interspecies_mating', 0.001)
        self.patience = kwargs.get('patience', float('inf'))
        self.champion_population_threshold = kwargs.get('champion_threshold', 0)


In [3]:
class Edge:
    """A gene object representing an edge in the neural network."""
    def __init__(self, weight):
        self.weight = weight
        self.enabled = True


class Node:
    """A gene object representing a node in the neural network."""
    def __init__(self, activation):
        self.output = 0
        self.bias = 0
        self.activation = activation


class Genome:
    """Base class for a standard genome used by the NEAT algorithm."""
    innov_edge = 0
    node_id = 2
    # map each (i,j) to an innovation number
    map_edge = {}
    
    # map each node to an edge used to when first created 
    map_node = {}
    
    def get_edge_key(innov):
        """Return the edge key"""
        for (i,j) in Genome.map_edge.keys():
            if Genome.map_edge[(i,j)] == innov:
                return (i,j)

    def get_node_id_by_edge(i, j):
        for k in Genome.map_node.keys():
            if Genome.map_node[k] == (i, j):
                return k
        return None

    def __init__(self, inputs, outputs, default_activation, distributions_parameters):
        # Nodes
        self._inputs = inputs
        self._outputs = outputs

        self._unhidden = inputs+outputs  # Number of unhidden nodes
        self._max_node = inputs+outputs  # Number of nodes (starting from i+o)

        # Structure
        self._edges = {}  # (i, j) : Edge
        self._nodes = {}  # NodeID : Node

        self._default_activation = default_activation

        # Performance
        self._fitness = 0
        self._adjusted_fitness = 0

        # range lim
        self.a = distributions_parameters['uniform_range']
        self.std = distributions_parameters['gauss_std']

    def generate(self):
        """
        Generate the neural network of this genome with minimal
        initial topology, i.e. (no hidden nodes). Call on genome
        creation.
        """
        # Minimum initial topology, no hidden layer

        # initialize nodes with default activation
        for n in range(self._max_node):
            self._nodes[n] = Node(self._default_activation)
            Genome.map_node[n] = None
        
        # connect all input nodes to all output nodes
        for i in range(self._inputs):
            for j in range(self._inputs, self._unhidden):
                self.add_edge(i, j, random.gauss(0, self.std))

    def forward(self, inputs):
        """
        Evaluate inputs and calculate the outputs of the
        neural network via the forward propagation algorithm.
        """

        # Check if the number of inputs is correct
        if len(inputs) != self._inputs:
            raise ValueError("Incorrect number of inputs.")

        # Set input values
        for i in range(self._inputs):
            self._nodes[i].output = inputs[i] # TODO might want to call activation function here

        # Generate backward-adjacency list (from -> to) a list of all incoming edges to a node
        
        _from = {}
        for n in self._nodes.keys():
            _from[n] = []

        for (i, j) in self._edges:
            if not self._edges[(i, j)].enabled:
                continue
            _from[j].append(i)

        ordered_nodes = itertools.chain(
            [n for n in self._nodes.keys() if n > self._unhidden ], # a list of all node ids that are hidden
            range(self._inputs, self._unhidden) 
        )

        # Calculate output values for each node in topological order
        for j in ordered_nodes:
            ax = 0  # Activation value
            for i in _from[j]:
                ax += self._edges[(i, j)].weight * self._nodes[i].output

            node = self._nodes[j]
            node.output = node.activation(ax + node.bias)

        return [self._nodes[n].output for n in range(self._inputs, self._unhidden)]

    def mutate(self, probabilities):
        """Randomly mutate the genome to initiate variation."""
        if self.is_disabled():
            self.add_enabled()

        # we extract probabilties use them to choose a key in acco
        # population = list(probabilities.keys())
        # weights = [probabilities[k] for k in population]
        # choice = random.choices(population, weights=weights)[0]

        if (random.random() < probabilities["node"]) | (len(self._nodes.keys())==self._unhidden):#"node": 
            self.add_node()
        if random.random() < probabilities["edge"]:
            (i, j) = self.random_pair()
            self.add_edge(i, j, random.uniform(-self.a, self.a))
        chance = random.random()
        if chance < (probabilities["weight_perturb"]+probabilities["weight_set"]): #choice == "weight_perturb" or choice == "weight_set":
            if chance < probabilities["weight_perturb"]:
                self.shift_weight("weight_perturb")
            else:
                self.shift_weight("weight_set")
        chance = random.random()
        if chance < (probabilities["bias_perturb"]+probabilities["bias_set"]):
            if chance < probabilities["bias_perturb"]:
                self.shift_bias("bias_perturb")
            else:
                self.shift_bias("bias_set")    
        # print(self._nodes.keys())
        
        self.reset()

    def add_node(self):
        """
        Add a new node between a randomly selected edge,
        disabling the parent edge.
        """
        enabled = [k for k in self._edges if self._edges[k].enabled]
        (i, j) = random.choice(enabled)
        edge = self._edges[(i, j)]
        edge.enabled = False

        # check if we already created a node for this edge
        # Genome.map_node : (node_id) : (i,j)
        if (i,j) in Genome.map_node.values():
            # if we already created a node for this edge, we use the same node_id
            new_node = Genome.get_node_id_by_edge(i, j)
        else:
            # if we haven't created a node for this edge
            # increment the node_id
            Genome.node_id += 1
            # use the new node_id as the node_id for the new node
            new_node = Genome.node_id
            Genome.map_node[new_node] = (i, j)

        self._nodes[new_node] = Node(self._default_activation)
        self.add_edge(i, new_node, 1.0)
        self.add_edge(new_node, j, edge.weight)
        return new_node

    def add_edge(self, i, j, weight):
        """Add a new connection between existing nodes."""
        if (i, j) in self._edges:
            self._edges[(i, j)].enabled = True
        else:
            self._edges[(i, j)] = Edge(weight)
            if (i, j) not in Genome.map_edge.keys():
                Genome.innov_edge += 1
                Genome.map_edge[(i, j)] = Genome.innov_edge

    def add_enabled(self):
        """Re-enable a random disabled edge."""
        disabled = [e for e in self._edges if not self._edges[e].enabled]

        if len(disabled) > 0:
            re_enabled = random.choice(disabled)
            self._edges[re_enabled].enabled = True
        

    def shift_weight(self, type):
        """Randomly shift, perturb, or set one of the edge weights."""
        e = random.choice(list(self._edges.keys()))
        if type == "weight_perturb":
            self._edges[e].weight += random.uniform(-self.a, self.a)
        elif type == "weight_set":
            self._edges[e].weight = random.uniform(-self.a, self.a)

    def shift_bias(self, type):
        """Randomly shift, perturb, or set the bias of an incoming edge."""
        # Select only nodes in the hidden and output layer
        # n = random.choice(range(self._inputs, self._max_node))
        # print([n for n in self._nodes.keys() if n >= self._inputs])
        # print(self._nodes.keys())
        n = random.choice([n for n in self._nodes.keys() if n >= self._inputs])
        if type == "bias_perturb":
            self._nodes[n].bias += random.uniform(-self.a, self.a)
        elif type == "bias_set":
            self._nodes[n].bias = random.uniform(-self.a, self.a)

    def random_pair(self):
        """
        Generate random nodes (i, j) such that:
        1. i is not an output
        2. j is not an input
        3. i != j
        """
        
        i = random.choice([n for n in self._nodes.keys() if not self.is_output(n)])
        j_list = [n for n in self._nodes.keys() if not self.is_input(n) and n != i]

        if not j_list:
            # j = self.node_id
            # self.add_node()
            j = self.add_node()
        else:
            j = random.choice(j_list)
        return (i, j)

    def is_input(self, n):
        """Determine if the node id is an input."""
        return 0 <= n < self._inputs

    def is_output(self, n):
        """Determine if the node id is an output."""
        return self._inputs <= n < self._unhidden

    def is_hidden(self, n):
        """Determine if the node id is hidden."""
        return self._unhidden <= n 

    def is_disabled(self):
        """Determine if all of its genes are disabled."""
        return all(self._edges[i].enabled == False for i in self._edges)
    
    def is_enabled(self):
        """Determine if all of its genes are enabled."""
        return all(self._edges[i].enabled == True for i in self._edges)

    def get_fitness(self):
        """Return the fitness of the genome."""
        return self._fitness

    def get_nodes(self):
        """Get the nodes of the network."""
        return self._nodes.copy()

    def get_edges(self):
        """Get the network's edges."""
        return self._edges.copy()

    # def get_num_nodes(self):
    #     """Get the number of nodes in the network."""
    #     return self._max_node

    def set_fitness(self, score):
        """Set the fitness score of this genome."""
        self._fitness = score

    def reset(self):
        """Reset the genome's internal state."""
        # for n in range(self._max_node):
        #     self._nodes[n].output = 0
        for n in self._nodes.values(): # {node_id: node}
            n.output = 0
        self._fitness = 0
    
    def create_nodes_from_edges(self):
        """Create nodes from edges"""
        # extract all nodes from edges
        node_ids= set([i for (i,j)in self._edges.keys()] + [j for (i,j) in self._edges.keys()])
        for node_id in node_ids:
            self._nodes[node_id] = Node(self._default_activation)
        
    def clone(self):
        """Return a clone of the genome.
        """
        return copy.deepcopy(self)
    
    def get_edges_innov(self):
        """Return the enovation number of the edges"""
        return [Genome.map_edge[(i,j)] for (i,j) in self._edges.keys()]
    
    def disable_edge(self, i, j):
        """Disable an edge"""
        self._edges[(i,j)].enabled = False
        
    def enable_edge(self, i, j):
        """Enable an edge"""
        self._edges[(i,j)].enabled = True
    
    def get_disabled_edges(self):
        """Return a list of disabled edges"""
        return [(i,j) for (i,j) in self._edges.keys() if not self._edges[(i,j)].enabled]

In [4]:
def genomic_distance(a :Genome, b:Genome, distance_weights): # distance_weights === c vector
    """Calculate the genomic distance between two genomes."""
    
    c1, c2, c3 = distance_weights['excess'], distance_weights['disjoint'] , distance_weights['weight']
    
    a_edges_innov = set(a.get_edges_innov())
    b_edges_innov = set(b.get_edges_innov())
    
    max_innov_a = max(a_edges_innov)
    max_innov_b = max(b_edges_innov)
    
    cut_off = min(max_innov_a, max_innov_b)
    
    Excess = 0
    Disjoint = 0
    for innov in a_edges_innov ^ b_edges_innov:
        if innov <= cut_off:
            Disjoint += 1
        else:
            Excess += 1      
    N = max(len(a_edges_innov), len(b_edges_innov))
    matching =len(a_edges_innov & b_edges_innov)
    
    W_avg = 0
    for i in a_edges_innov & b_edges_innov:
        key = Genome.get_edge_key(i)
        W_avg += abs(a._edges[key].weight - b._edges[key].weight)
    W_avg /= matching
    dist= (c1 * Excess/N + c2 * Disjoint/N + c3 * W_avg)
    # print(f'a_nodes: {a._nodes.keys()} , b_nodes: {b._nodes.keys()} , Excess: {Excess} , Disjoint: {Disjoint} , N: {N} , matching: {matching} , W_avg: {W_avg} , dist: {dist}')
    return dist
    

def genomic_crossover(a:Genome, b :Genome):
    """Breed two genomes and return the child. Matching genes
    are inherited randomly, while disjoint genes are inherited
    from the fitter parent.
    """
    # Template genome for child
    distributions_parameters = {
        "uniform_range" : a.a,
        "gauss_std": a.std
    }
    child = Genome(a._inputs, a._outputs, a._default_activation, distributions_parameters)
    
    # extract edges (by innov number) from each parent
    a_edges_innov =set(a.get_edges_innov())
    b_edges_innov = set(b.get_edges_innov())
    
    disabled_a = a.get_disabled_edges()
    disabled_b = b.get_disabled_edges()
    
    # disabled edges in both parents
    disabled_edges = set(disabled_a) ^ set(disabled_b)
    
    # Inherit homologous gene from a random parent
    for innov in a_edges_innov & b_edges_innov:
        parent = random.choice([a, b])
        key = Genome.get_edge_key(innov)
        if key in disabled_edges:
            child._edges[key] = copy.deepcopy(parent._edges[key])
            is_disabled = random.random()<0.75 #choice([True, False], p=[0.75, 0.25])
            if is_disabled:
                child._edges[key].enabled = False
            else:
                child._edges[key].enabled = True
        child._edges[key] = copy.deepcopy(parent._edges[key])
        
        
    # inherit disjoint genes from either parent
    max_innov_a = max(a_edges_innov)
    max_innov_b = max(b_edges_innov)
    
    cut_off = min(max_innov_a, max_innov_b)
    
    # Inherit disjoint genes from the parent that has them
    for innov in a_edges_innov ^ b_edges_innov:
        if innov <= cut_off:
            key = Genome.get_edge_key(innov)
            if innov in a_edges_innov:
                child._edges[key] = copy.deepcopy(a._edges[key])
            else:
                child._edges[key] = copy.deepcopy(b._edges[key])
    
    
    # inherit excess genes from the fitter parent
    fitter = a if a._fitness > b._fitness else b
    for innov in fitter.get_edges_innov():
        if innov > cut_off:
            key = Genome.get_edge_key(innov)
            child._edges[key] = copy.deepcopy(fitter._edges[key])
    child.create_nodes_from_edges()
    child.reset()
    return child

In [5]:
class Specie(object):
    """
    A specie represents a set of genomes whose genomic distances 
    between them fall under the FeedForwardNet's delta threshold.
    """

    def __init__(self, max_fitness_history, *members):
        self._members :list[Genome] = list(members)
        self._fitness_history = []
        self._fitness_sum = 0
        self._max_fitness_history = max_fitness_history

    def breed(self, mutation_probabilities, breed_probabilities,other_specie=None):
        """Return a child as a result of either a mutated clone
        or crossover between two parent genomes.
        """
        # Either mutate a clone or breed two random genomes
        population = list(breed_probabilities.keys())
        probabilities = [breed_probabilities[k] for k in population]
        choice = random.choices(population, weights=probabilities)[0]

        if choice == "asexual" or len(self._members) == 1:
            child = random.choice(self._members).clone()
            child.mutate(mutation_probabilities)
        elif choice == "sexual":
            if other_specie!=None:
                # Interspecies mating
                mom = random.choice(self._members)
                dad = random.choice(other_specie._members)
                child = genomic_crossover(mom, dad)
            else:
                (mom, dad) = random.sample(self._members, 2)
                child = genomic_crossover(mom, dad)
        return child

    def update_fitness(self):
        """Update the adjusted fitness values of each genome 
        and the historical fitness."""
        for g in self._members:
            g._adjusted_fitness = g._fitness/len(self._members)
        self._fitness_sum = sum([g._adjusted_fitness for g in self._members])
        self._fitness_history.append(self._fitness_sum)
        # TODO: check if it pops the oldest fitness and not the newest
        if len(self._fitness_history) > self._max_fitness_history:
            self._fitness_history.pop(0)

    def cull_genomes(self, fittest_only):
        """Exterminate the weakest genomes per specie."""
        self._members.sort(key=lambda g: g._fitness, reverse=True)
        if fittest_only:
            # Only keep the winning genome
            remaining = 1
        else:
            # Keep top 25%
            # TODO: try to change to 50%
            # TODO: check what they did in original paper
            remaining = int(math.ceil(0.5*len(self._members)))

        self._members = self._members[:remaining]

    def get_best(self):
        """Get the member with the highest fitness score."""
        return max(self._members, key=lambda g: g._fitness)

    def can_progress(self):
        """Determine whether species should survive the culling."""
        n = len(self._fitness_history)
        avg = sum(self._fitness_history) / n
        # TODO: patience = self._max_fitness_history
        return avg > self._fitness_history[0] or n < self._max_fitness_history

In [6]:
class NEAT(object):
    """Base class for 'NEAT' algorithm that learns through the evolution
    of a population of genomes.
    """

    def __init__(self, inputs, outputs, population=100, hyperparams=Hyperparameters()):
        self._inputs = inputs
        self._outputs = outputs

        self._species :list[Specie] = []
        self._population = population

        # Hyper-parameters
        self._hyperparams = hyperparams

        self._generation = 0
        self._current_species = 0
        self._current_genome = 0

        self._global_best = None
        self._champions = []

    def generate(self):
        """Generate the initial population of genomes."""
        for i in range(self._population):
            g = Genome(self._inputs, self._outputs,
                       self._hyperparams.default_activation,
                       self._hyperparams.distributions_parameters
                       )
            g.generate()
            self.classify_genome(g)

        # Set the initial best genome
        self._global_best = self._species[0]._members[0]

    def classify_genome(self, genome):
        """Classify genomes into species via the genomic
        distance algorithm.
        """
        if not self._species:
            # Empty population
            self._species.append(Specie(
                self._hyperparams.max_fitness_history, genome
            )
            )
        else:
            # Compare genome against representative s[0] in each specie
            for s in self._species:
                rep = s._members[0]
                dist = genomic_distance(
                    genome, rep, self._hyperparams.distance_weights
                )
                if dist <= self._hyperparams.delta_threshold:
                    s._members.append(genome)
                    return
            # Doesn't fit with any other specie, create a new one
            self._species.append(Specie(
                self._hyperparams.max_fitness_history, genome
            )
            )

    def update_fittest(self):
        """Update the highest fitness score of the whole population."""
        top_performers = [s.get_best() for s in self._species]
        current_top = max(top_performers, key=lambda g: g._fitness)
        self._champions = [s.get_best() for s in self._species if len(s._members) > self._hyperparams.champion_population_threshold]

        # TODO: roll over the champion automatically to the next generation if specie has enough members
        if current_top._fitness > self._global_best._fitness:
            self._global_best = current_top.clone()
        

    def evolve(self):
        """
        Evolve the population by eliminating the poorest performing
        genomes and repopulating with mutated children, prioritizing
        the most promising species.
        """
        global_fitness_sum = 0
        for s in self._species:
            s.update_fitness()
            global_fitness_sum += s._fitness_sum
        
        if global_fitness_sum == 0:
            # No progress, mutate everybody
            for s in self._species:
                for g in s._members:
                    g.mutate(self._hyperparams.mutation_probabilities)
        else:
            # Only keep the species with potential to improve
            surviving_species = []
            for s in self._species:
                if s.can_progress():
                    surviving_species.append(s)
            self._species = surviving_species

            # Eliminate lowest performing genomes per specie
            for s in self._species:
                s.cull_genomes(False)
            
            for c in self._champions:
                c_clone = c.clone()
                self.classify_genome(c_clone)
        

            # Repopulate
            for i, s in enumerate(self._species):
                ratio = s._fitness_sum/global_fitness_sum
                diff = self._population - self.get_population()
                offspring = int(round(ratio * diff))
                
                # Repopulate : each species gets to mutate and breed in proportion to its fitness
                for j in range(offspring):
                    s1= None
                    if len(self._species)>1 and random.random() < self._hyperparams.interspecies_mating:
                        # Interspecies mating
                        s1 = random.choice(list(set(self._species)-set([s])))
                    self.classify_genome(
                        s.breed(
                            self._hyperparams.mutation_probabilities,
                            self._hyperparams.breed_probabilities,
                            s1
                        )
                    )

            if not self._species:
                print("No species survived, repopulating...")
                for i in range(self._population):
                    
                    if i % 3 == 0:
                        g = self._global_best.clone()
                    else:
                        g = Genome(self._inputs, self._outputs,
                                   self._hyperparams.default_activation, 
                                   self._hyperparams.distributions_parameters)
                        g.generate()
                    g.mutate(self._hyperparams.mutation_probabilities)
                    self.classify_genome(g)
        
        # print(f"Generation {self._generation}, #Species: {len(self._species)}, #Population: {self.get_population()}")
        self._generation += 1

    def should_evolve(self):
        """Determine if the system should continue to evolve
        based on the maximum fitness and generation count.
        """
        self.update_fittest()
        fit = self._global_best._fitness <= self._hyperparams.max_fitness
        end = self._generation != self._hyperparams.max_generations

        return fit and end

    def next_iteration(self):
        """Call after every evaluation of individual genomes to
        progress training.
        """
        s = self._species[self._current_species]
        if self._current_genome < len(s._members)-1:
            self._current_genome += 1
        else:
            if self._current_species < len(self._species)-1:
                self._current_species += 1
                self._current_genome = 0
            else:
                # Evolve to the next generation
                self.evolve()
                self._current_species = 0
                self._current_genome = 0
    
    def evaluate_parallel(self, evaluator, *args, **kwargs):
        """Evaluate the entire population on separate processes
        to progress training. The evaluator function must take a Genome
        as its first parameter and return a numerical fitness score.

        Any global state passed to the evaluator is copied and will not
        be modified at the parent process.
        """
        max_proc = max(mp.cpu_count()-1, 1)
        pool = mp.Pool(processes=max_proc)

        results = {}
        for i in range(len(self._species)):
            for j in range(len(self._species[i]._members)):
                results[(i, j)] = pool.apply_async(
                    evaluator,
                    args=[self._species[i]._members[j]]+list(args),
                    kwds=kwargs
                )

        for key in results:
            genome = self._species[key[0]]._members[key[1]]
            genome.set_fitness(results[key].get())

        pool.close()
        pool.join()
        self.evolve()

    def get_fittest(self):
        """Return the genome with the highest global fitness score."""
        return self._global_best

    def get_population(self):
        """Return the true (calculated) population size."""
        return sum([len(s._members) for s in self._species])

    def get_current(self):
        """Get the current genome for evaluation."""
        s = self._species[self._current_species]
        return s._members[self._current_genome]

    def get_current_species(self):
        """Get index of current species being evaluated."""
        return self._current_species

    def get_current_genome(self):
        """Get index of current genome being evaluated."""
        return self._current_genome

    def get_generation(self):
        """Get the current generation number of this population."""
        return self._generation

    def get_species(self):
        """Get the list of species and their respective member genomes."""
        return self._species

    def save(self, filename):
        """Save an instance of the population to disk."""
        with open(filename+'.neat', 'wb') as _out:
            pickle.dump(self, _out, pickle.HIGHEST_PROTOCOL)

    @staticmethod
    def load(filename):
        """Return an instance of a population from disk."""
        with open(filename+'.neat', 'rb') as _in:
            return pickle.load(_in)

In [ ]:
def fitness(expected, output):
    """Calculates the similarity score between expected and output."""
    s = 0
    for i in range(4):
        s += (expected[i] - output[i])**2
    return 1/(1 + math.sqrt(s))


def evaluate(genome):
    """Evaluates the current genome."""
    f1 = genome.forward([0.0, 0.0])[0]
    f2 = genome.forward([1.0, 0.0])[0]
    f3 = genome.forward([0.0, 1.0])[0]
    f4 = genome.forward([1.0, 1.0])[0]
    return fitness([0.0, 1.0, 1.0, 0.0], [f1, f2, f3, f4])

In [39]:
class Simulation:
    def __init__(self ,inputs: int, outputs: int, population: int, hyperparams: dict, fitness_function: Callable, eval_function: Callable, verbose: bool = False):
        self.inputs = inputs
        self.outputs = outputs
        self.population = population
        self.hyperparams = Hyperparameters(**hyperparams)
        self.fitness_function = fitness_function
        self.eval_function = eval_function
        self.ffn = None  # FeedForward Network instance
        self.verbose = verbose 
        self.hist = np.array([0,0,0,0,0])

    def reset_network(self):
        """Initialize the FeedForward Network with the specified hyperparameters."""
        self.ffn = NEAT(self.inputs, self.outputs, self.population, self.hyperparams)
        self.ffn.generate()

    def run_simulation(self):
        """Run the simulation, evolving the network through generations."""
        self.elapsed_generations = 0
        print("Training...")
        while self.ffn.should_evolve():
            self.ffn.evaluate_parallel(self.eval_function)
            # Print training progress
            current_gen = self.ffn.get_generation()
            current_best = self.ffn.get_fittest()
            results = np.array([current_gen, current_best.get_fitness(), len(self.ffn._species), len([e for e in current_best.get_edges().values() if not e.enabled]), len(current_best.get_nodes()) - 3]) 
            self.hist = np.vstack([self.hist, results])
      
            if self.verbose:
                print(f"Current Fittness: {current_best.get_fitness() * 100:.2f}% | Generation {current_gen}/{self.hyperparams.max_generations}")
                if current_best.get_fitness() >= self.hyperparams.max_fitness:
                    break
            
            self.elapsed_generations += 1

    def display_results(self):
        """Evaluate and display the results of the best genome after the simulation."""
        best = self.ffn.get_fittest()
        fit = self.eval_function(best)
        if self.verbose:
            print("\nEvaluation of Best Network:")
            print(f"Overall Fitness: {fit * 100:.2f}%")

    def evaluate_best_network(self, test_inputs, expected_outputs):
        """
        Evaluate the best network with a set of test inputs and compare against expected outputs.
        Parameters:
            test_inputs (list): A list of inputs to test the network with.
            expected_outputs (list): The expected outputs for each input.
        """
        best = self.ffn.get_fittest()
        actual_outputs = [best.forward(input_data)[0] for input_data in test_inputs]

        # Assuming self.fitness_function can compare a list of expected and actual outputs
        fit = self.fitness_function(expected_outputs, actual_outputs)

        print("\nEvaluation of Best Network:")
        for i, input_data in enumerate(test_inputs):
            print(f"Input: {input_data} | Expected Output: {expected_outputs[i]} | Actual Output: {actual_outputs[i]:.3f}")
        print(f"\nOverall Fitness: {fit * 100:.2f}%")

    def get_df(self):
        return pd.DataFrame(self.hist,columns=["generation", "fitness", "species", "active", "hidden nodes"])

    def display_best_network_structure(self):
        """
        Displays the structure of the best network in the population.
        """
        best = self.ffn.get_fittest()
        edges = best.get_edges()
        nodes = best.get_nodes()

        graph = {}
        for (i, j), edge in edges.items():
            if not edge.enabled:
                continue
            if i not in graph:
                graph[i] = []
            graph[i].append(j)

        num_inputs = self.inputs
        num_outputs = self.outputs
        num_hidden = len(nodes) - (num_inputs + num_outputs)
        num_active_connections = sum(1 for edge in edges.values() if edge.enabled)
        num_inactive_connections = sum(1 for edge in edges.values() if not edge.enabled)
        if self.verbose:
            print("\nBest Network Structure:")
            print(f"# of Hidden Nodes: {num_hidden}")
            print(f"# of Active Connections: {num_active_connections}")

            print("\nBest Network Structure:")
            for k in sorted(graph):
                print(f"{k} -> {graph[k]}")

        return num_hidden, num_active_connections, len(self.ffn._species), self.elapsed_generations, best._fitness

## Baseline

In [11]:
runs = 1
hyperparams = {
    "max_generations": 1000,  # how many generations we run before aborting
    "max_fitness": 3.9/4,  # target fitness for early stopping
    "default_activation": activations.sigmoid,  # 1/(1+math.exp(-x))
    "breed_probabilities": {'sexual': 0.75, 'asexual': 0.25},
    "mutation_probabilities": {
        # note: default a = 2
        'node': 0.03,
        'edge': 0.1,
        'weight_perturb': 0.8*0.9,
        'weight_set': 0.8*0.1,
        'bias_perturb': 0.12*0.3,
        'bias_set': 0.12*0.7
    },
    "distance_weights": {
        'excess': 1.0,  # c1
        'disjoint': 1.0,  # c2
        'weight': 0.4  # c3 
    },
    "interspecies_mating": 0.001,
    "max_fitness_history": 15,  # how many generations to wait for fitness to improve before eliminating a species
    # min number of members in a species to clone champion to next generation
    "champion_population_threshold": 5,
    "delta_threshold": 1,  # threshold for speciation
    "distributions_parameters": {
        "uniform_range" : 1, # uniform [-a, a]
        "gauss_std": 1/5
    }
}


def fitness(expected, output):
    """Calculates the similarity score between expected and output."""
    s = 0
    for i in range(4):
        s += (expected[i] - output[i])**2
    return 1/(1 + math.sqrt(s))


def evaluate(genome):
    """Evaluates the current genome."""
    f1 = genome.forward([0.0, 0.0])[0]
    f2 = genome.forward([1.0, 0.0])[0]
    f3 = genome.forward([0.0, 1.0])[0]
    f4 = genome.forward([1.0, 1.0])[0]
    return fitness([0.0, 1.0, 1.0, 0.0], [f1, f2, f3, f4])


In [12]:
sim_xor = Simulation(
            inputs=2,
            outputs=1,
            population=150,
            hyperparams=hyperparams,
            fitness_function=fitness,
            eval_function=evaluate,
            verbose=False
        )
# store results :{num_hidden, active_connections, #species ,generations_to_solution, best_fitness}
metrics = []
start = time.time()
for i in range(runs):
    sim_xor.reset_network()
    sim_xor.run_simulation()
    sim_xor.display_results()
    results = sim_xor.display_best_network_structure()
    metrics.append(results)
    num_hidden, active_connections, species ,generations_to_solution, best_fitness = results
    print(f"Run {i+1:>3} complete | Best accuracy: {int(best_fitness*100):>3}%, {num_hidden} hidden nodes, {active_connections:>3} active connections, species: {species:>3}) ,{generations_to_solution:>3} generations to solution | {time.time() - start:.2f} seconds elapsed.")

df = pd.DataFrame(metrics, columns=[
                "num_hidden", "active_connections", "species" ,"generations_to_solution", "best_fitness"])
df['best_fitness'] = df['best_fitness'] * 100
print("\nSummary (all):")
display(df.describe())
print("\nSummary (successful runs):")
display(df[df['generations_to_solution'] < hyperparams['max_generations']].describe())
end = time.time() - start
print(f"Time took: {end:,} seconds.")

Training...


Run   1 complete | Best accuracy:  97%, 3 hidden nodes,   9 active connections, species:   1) ,149 generations to solution | 35.83 seconds elapsed.

Summary (all):


,num_hidden,active_connections,species,generations_to_solution,best_fitness
count,1.0,1.0,1.0,1.0,1.000000
mean,3.0,9.0,1.0,149.0,97.546622
std,NaN,NaN,NaN,NaN,NaN
min,3.0,9.0,1.0,149.0,97.546622
25%,3.0,9.0,1.0,149.0,97.546622
50%,3.0,9.0,1.0,149.0,97.546622
75%,3.0,9.0,1.0,149.0,97.546622
max,3.0,9.0,1.0,149.0,97.546622



Summary (successful runs):


,num_hidden,active_connections,species,generations_to_solution,best_fitness
count,1.0,1.0,1.0,1.0,1.000000
mean,3.0,9.0,1.0,149.0,97.546622
std,NaN,NaN,NaN,NaN,NaN
min,3.0,9.0,1.0,149.0,97.546622
25%,3.0,9.0,1.0,149.0,97.546622
50%,3.0,9.0,1.0,149.0,97.546622
75%,3.0,9.0,1.0,149.0,97.546622
max,3.0,9.0,1.0,149.0,97.546622


Time took: 35.97653412818909 seconds.


In [13]:
runs = 100
hyperparams = {
    "max_generations": 400,  # how many generations we run before aborting
    "max_fitness": 3.9/4,  # target fitness for early stopping
    "default_activation": activations.sigmoid,  # 1/(1+math.exp(-x))
    "breed_probabilities": {'sexual': 0.75, 'asexual': 0.25},
    "mutation_probabilities": {
        # note: default a = 2
        'node': 0.03,
        'edge': 0.05,
        'weight_perturb': 0.8*0.9,
        'weight_set': 0.8*0.1,
        'bias_perturb': 0.12*0.3,
        'bias_set': 0.12*0.7
    },
    "distance_weights": {
        'excess': 1.0,  # c1
        'disjoint': 1.0,  # c2
        'weight': 0.4  # c3 
    },
    "interspecies_mating": 0.001,
    "max_fitness_history": 15,  # how many generations to wait for fitness to improve before eliminating a species
    # min number of members in a species to clone champion to next generation
    "champion_population_threshold": 5,
    "delta_threshold": 3.0,  # threshold for speciation
    "distributions_parameters": {
        "uniform_range" : 1, # uniform [-a, a]
        "gauss_std": 1/5
    }
}


def fitness(expected, output):
    """Calculates the similarity score between expected and output."""
    s = 0
    for i in range(4):
        s += (expected[i] - output[i])**2
    return 1/(1 + math.sqrt(s))


def evaluate(genome):
    """Evaluates the current genome."""
    f1 = genome.forward([0.0, 0.0])[0]
    f2 = genome.forward([1.0, 0.0])[0]
    f3 = genome.forward([0.0, 1.0])[0]
    f4 = genome.forward([1.0, 1.0])[0]
    return fitness([0.0, 1.0, 1.0, 0.0], [f1, f2, f3, f4])


In [14]:
sim_xor = Simulation(
            inputs=2,
            outputs=1,
            population=150,
            hyperparams=hyperparams,
            fitness_function=fitness,
            eval_function=evaluate,
            verbose=True
        )
# store results :{num_hidden, active_connections, #species ,generations_to_solution, best_fitness}
metrics = []
start = time.time()
for i in range(runs):
    sim_xor.reset_network()
    sim_xor.run_simulation()
    sim_xor.display_results()
    results = sim_xor.display_best_network_structure()
    metrics.append(results)
    num_hidden, active_connections, species ,generations_to_solution, best_fitness = results
    print(f"Run {i+1:>3} complete | Best accuracy: {int(best_fitness*100):>3}%, {num_hidden} hidden nodes, {active_connections:>3} active connections, species: {species:>3}) ,{generations_to_solution:>3} generations to solution | {time.time() - start:.2f} seconds elapsed.")

df = pd.DataFrame(metrics, columns=[
                "num_hidden", "active_connections", "species" ,"generations_to_solution", "best_fitness"])
df['best_fitness'] = df['best_fitness'] * 100
print("\nSummary (all):")
print(df.describe())
print("\nSummary (successful runs):")
print(df[df['generations_to_solution'] < hyperparams['max_generations']].describe())
end = time.time() - start
print(f"Time took: {end:,} seconds.")

Training...


Current Fittness: 44.72% | Generation 1/400
Current Fittness: 49.99% | Generation 2/400
Current Fittness: 50.00% | Generation 3/400
Current Fittness: 50.00% | Generation 4/400
Current Fittness: 50.00% | Generation 5/400
Current Fittness: 50.00% | Generation 6/400
Current Fittness: 50.00% | Generation 7/400
Current Fittness: 50.00% | Generation 8/400
Current Fittness: 50.00% | Generation 9/400
Current Fittness: 50.00% | Generation 10/400
Current Fittness: 50.00% | Generation 11/400
Current Fittness: 50.00% | Generation 12/400
Current Fittness: 50.00% | Generation 13/400
Current Fittness: 50.00% | Generation 14/400
Current Fittness: 50.00% | Generation 15/400
Current Fittness: 50.00% | Generation 16/400
Current Fittness: 50.00% | Generation 17/400
Current Fittness: 50.08% | Generation 18/400
Current Fittness: 50.08% | Generation 19/400
Current Fittness: 50.08% | Generation 20/400
Current Fittness: 50.09% | Generation 21/400
Current Fittness: 50.09% | Generation 22/400
Current Fittness: 5

KeyboardInterrupt: 

## Experimentations and improvements

in all experaments we do **one run** with **"max_generations": 1000**

### rapid mutation Edge

here we changed **('edge': 0.08)**<br>
also by trial and error we came to the concostion that **(delta_threshold": 1.5)** works better

In [ ]:
runs = 1
hyperparams = {
    "max_generations": 1000,  # how many generations we run before aborting
    "max_fitness": 3.9/4,  # target fitness for early stopping
    "default_activation": activations.sigmoid,  # 1/(1+math.exp(-x*4.9))
    "breed_probabilities": {'sexual': 0.75, 'asexual': 0.25},
    "mutation_probabilities": {
        # note: default a = 2
        'node': 0.3,
        'edge': 0.1, # now the edge wait would change more rapidly
        'weight_perturb': 0.8*0.9,
        'weight_set': 0.8*0.1,
        'bias_perturb': 0.12*0.3,
        'bias_set': 0.12*0.7
    },
    "distance_weights": {
        'excess': 1.0,  # c1
        'disjoint': 1.0,  # c2
        'weight': 0.4  # c3 
    },
    "interspecies_mating": 0.001,
    "max_fitness_history": 15,  # how many generations to wait for fitness to improve before eliminating a species
    # min number of members in a species to clone champion to next generation
    "champion_population_threshold": 5,
    "delta_threshold": 1.5,  # threshold for speciation
    "distributions_parameters": {
        "uniform_range" : 1, # uniform [-a, a]
        "gauss_std": 1/5
    }
}

In [45]:
sim_xor = Simulation(
            inputs=2,
            outputs=1,
            population=150,
            hyperparams=hyperparams,
            fitness_function=fitness,
            eval_function=evaluate,
            verbose=True
        )
# store results :{num_hidden, active_connections, #species ,generations_to_solution, best_fitness}
metrics = []
start = time.time()
for i in range(runs):
    sim_xor.reset_network()
    sim_xor.run_simulation()
    sim_xor.display_results()
    results = sim_xor.display_best_network_structure()
    metrics.append(results)
    num_hidden, active_connections, species ,generations_to_solution, best_fitness = results
    print(f"Run {i+1:>3} complete | Best accuracy: {int(best_fitness*100):>3}%, {num_hidden} hidden nodes, {active_connections:>3} active connections, species: {species:>3}) ,{generations_to_solution:>3} generations to solution | {time.time() - start:.2f} seconds elapsed.")

df_ = pd.DataFrame(metrics, columns=[
                "num_hidden", "active_connections", "species" ,"generations_to_solution", "best_fitness"])
df['best_fitness'] = df['best_fitness'] * 100
print("\nSummary (all):")
print(df.describe())
print("\nSummary (successful runs):")
print(df[df['generations_to_solution'] < hyperparams['max_generations']].describe())
end = time.time() - start
print(f"Time took: {end:,} seconds.")

Training...


Current Fittness: 48.56% | Generation 1/1000
Current Fittness: 49.97% | Generation 2/1000
Current Fittness: 50.00% | Generation 3/1000
Current Fittness: 50.10% | Generation 4/1000
Current Fittness: 50.10% | Generation 5/1000
Current Fittness: 50.13% | Generation 6/1000
Current Fittness: 50.16% | Generation 7/1000
Current Fittness: 50.16% | Generation 8/1000
Current Fittness: 50.16% | Generation 9/1000
Current Fittness: 50.20% | Generation 10/1000
Current Fittness: 50.20% | Generation 11/1000
Current Fittness: 50.20% | Generation 12/1000
Current Fittness: 50.20% | Generation 13/1000
Current Fittness: 50.22% | Generation 14/1000
Current Fittness: 50.23% | Generation 15/1000
Current Fittness: 50.23% | Generation 16/1000
Current Fittness: 50.23% | Generation 17/1000
Current Fittness: 50.24% | Generation 18/1000
Current Fittness: 50.24% | Generation 19/1000
Current Fittness: 50.24% | Generation 20/1000
Current Fittness: 50.41% | Generation 21/1000
Current Fittness: 50.41% | Generation 22/10

In [47]:
sim_xor.get_df().to_csv("results.csv")